# 6.3 - MNIST benchmark analysis

Quantitative and qualitative comparison of the MNIST benchmark across `Nearest Neighbor`, `DiCE`, `Growing Spheres`, `FACE`, and `CertCF`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")

RESULT_PATH = Path("../results/benchmark_mnist.parquet")
METHOD_LABELS = {
    "nn": "Nearest Neighbor",
    "dice": "DiCE",
    "gs": "Growing Spheres",
    "face": "FACE",
    "certcf": "Our method",
}


## 1. Load and overview

The benchmark stores one row per `(source image, target digit, method)` task. For MNIST we evaluate every selected source image against all 9 alternative target digits.

In [ ]:
if not RESULT_PATH.exists():
    raise FileNotFoundError(f"Missing benchmark file: {RESULT_PATH}")

df = pd.read_parquet(RESULT_PATH).copy()
df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])
df["source_class"] = df["source_class"].astype("Int64") if "source_class" in df.columns else df["y_orig"].astype("Int64")
df["target_class"] = df["target_class"].astype("Int64")
if "y_true" in df.columns:
    df["y_true"] = df["y_true"].astype("Int64")

method_order = [m for m in METHOD_LABELS.values() if m in df["method_label"].unique()]
x_orig_cols = sorted([c for c in df.columns if c.startswith("x_orig_")], key=lambda c: int(c.split("_")[-1]))
x_cf_cols = sorted([c for c in df.columns if c.startswith("x_cf_")], key=lambda c: int(c.split("_")[-1]))

overview = pd.DataFrame({
    "n_rows": [len(df)],
    "n_methods": [df["method_label"].nunique()],
    "n_source_images": [df[["query_idx"]].drop_duplicates().shape[0]],
    "n_tasks": [df[["query_idx", "target_class"]].drop_duplicates().shape[0]],
    "n_target_digits": [df["target_class"].nunique()],
})
display(overview)
display(df.groupby("method_label").size().rename("rows").to_frame())


## 2. Coverage

Before comparing methods, we check that the task generation covers the source and target digits as expected.

In [ ]:
coverage = (
    df[["query_idx", "source_class", "target_class"]]
    .drop_duplicates()
    .groupby(["source_class", "target_class"])
    .size()
    .unstack(fill_value=0)
)
display(coverage.style.background_gradient(cmap="Blues", axis=0))

plt.figure(figsize=(8, 6))
sns.heatmap(coverage, annot=True, fmt="d", cmap="Blues")
plt.title("Task coverage by source digit and target digit")
plt.xlabel("Target digit")
plt.ylabel("Source digit")
plt.show()


## 3. Validity

Validity measures whether the returned counterfactual is classified as the desired target digit. This is the first metric to read: proximity and sparsity are only meaningful on valid counterfactuals.

In [ ]:
validity_summary = (
    df.groupby("method_label")["success"]
    .agg(validity_pct=lambda s: 100.0 * s.mean(), n_tasks="size")
    .reindex(method_order)
)
display(validity_summary.style.format({"validity_pct": "{:.1f}%", "n_tasks": "{:d}"}))

validity_by_target = (
    df.groupby(["method_label", "target_class"])["success"]
    .mean()
    .mul(100.0)
    .unstack("target_class")
    .reindex(method_order)
)
display(validity_by_target.style.format("{:.1f}%").background_gradient(cmap="YlGn", axis=0))

plt.figure(figsize=(9, 4.5))
sns.heatmap(validity_by_target, annot=True, fmt=".1f", cmap="YlGn", vmin=0, vmax=100)
plt.title("Validity by target digit (%)")
plt.xlabel("Target digit")
plt.ylabel("")
plt.show()


In [ ]:
def validity_proximity_curve(distances, eps_grid=None, n_thresholds=300):
    distances = np.asarray(distances, dtype=float)
    finite = np.sort(distances[np.isfinite(distances)])
    max_finite = float(finite.max()) if finite.size else 0.0
    if eps_grid is None:
        eps_grid = np.linspace(0.0, max_finite, n_thresholds) if max_finite > 0 else np.array([0.0])
    curve = np.searchsorted(finite, eps_grid, side="right") / len(distances)
    auc = np.trapezoid(curve, eps_grid) / eps_grid[-1] if eps_grid.size > 1 and eps_grid[-1] > 0 else float(curve[-1])
    return eps_grid, 100.0 * curve, auc

curve_rows = []
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in zip(axes, ["l2_distance", "l1_distance"], ["L2", "L1"]):
    finite_all = df.loc[df["success"], metric].to_numpy(dtype=float)
    finite_all = finite_all[np.isfinite(finite_all)]
    eps_grid = np.linspace(0.0, float(finite_all.max()), 300) if finite_all.size else np.array([0.0])
    for method in method_order:
        sub = df[df["method_label"] == method]
        distances = sub[metric].to_numpy(dtype=float)
        distances[~sub["success"].to_numpy(dtype=bool)] = np.inf
        distances[~np.isfinite(distances)] = np.inf
        x, y, auc = validity_proximity_curve(distances, eps_grid=eps_grid)
        curve_rows.append({"metric": title, "method": method, "auc": auc})
        ax.step(x, y, where="post", linewidth=2, label=method)
    ax.set_title(f"Validity as a function of the allowed {title} distance")
    ax.set_xlabel(f"{title} distance budget")
    ax.set_ylabel("Validity (%)")
    ax.set_ylim(0, 105)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

curve_auc = pd.DataFrame(curve_rows).pivot(index="method", columns="metric", values="auc").reindex(method_order)
display(curve_auc.style.format("{:.3f}").background_gradient(cmap="Blues", axis=0))


## 4. Proximity

Proximity is computed only on valid counterfactuals. Lower values indicate smaller changes from the original image.

In [ ]:
df_ok = df[df["success"]].copy()

proximity_summary = (
    df_ok.groupby("method_label")
    .agg(
        mean_l2=("l2_distance", "mean"),
        std_l2=("l2_distance", "std"),
        mean_l1=("l1_distance", "mean"),
        std_l1=("l1_distance", "std"),
        n_valid=("success", "size"),
    )
    .reindex(method_order)
)
display(
    proximity_summary.style.format({
        "mean_l2": "{:.3f}",
        "std_l2": "{:.3f}",
        "mean_l1": "{:.3f}",
        "std_l1": "{:.3f}",
        "n_valid": "{:d}",
    })
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df_ok, x="method_label", y="l2_distance", order=method_order, ax=axes[0])
axes[0].set_title("L2 proximity on valid counterfactuals")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)
sns.boxplot(data=df_ok, x="method_label", y="l1_distance", order=method_order, ax=axes[1])
axes[1].set_title("L1 proximity on valid counterfactuals")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## 5. Sparsity and runtime

Sparsity is also computed on valid counterfactuals only. Runtime is separated into build time and query time.

In [ ]:
sparsity_summary = (
    df_ok.groupby("method_label")
    .agg(
        mean_sparsity=("l0_sparsity", lambda s: 100.0 * s.mean()),
        std_sparsity=("l0_sparsity", lambda s: 100.0 * s.std()),
        n_valid=("success", "size"),
    )
    .reindex(method_order)
)
display(sparsity_summary.style.format({"mean_sparsity": "{:.1f}%", "std_sparsity": "{:.1f}%", "n_valid": "{:d}"}))

runtime_summary = (
    df.groupby("method_label")
    .agg(
        build_time_s=("build_time_s", "first"),
        mean_query_time_s=("runtime_s", "mean"),
        total_query_time_s=("runtime_s", "sum"),
    )
    .reindex(method_order)
)
display(runtime_summary.style.format("{:.3f}").background_gradient(cmap="Oranges", axis=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
runtime_summary["build_time_s"].plot(kind="bar", ax=axes[0], color=sns.color_palette("tab10", len(runtime_summary)))
axes[0].set_title("Build time")
axes[0].set_ylabel("seconds")
axes[0].tick_params(axis="x", rotation=25)
runtime_summary["mean_query_time_s"].plot(kind="bar", ax=axes[1], color=sns.color_palette("tab10", len(runtime_summary)))
axes[1].set_title("Mean query time")
axes[1].set_ylabel("seconds")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## 6. Joint validity-closeness score

We summarize the reliability-closeness trade-off with a single score. Failed queries contribute zero, and valid counterfactuals contribute more when they stay close to the source image.

In [ ]:
def validity_closeness_score(distances, budget):
    distances = np.asarray(distances, dtype=float)
    distances = np.where(np.isfinite(distances), distances, np.inf)
    return np.mean(np.maximum(1.0 - distances / budget, 0.0))

finite_l2 = df.loc[df["success"], "l2_distance"].to_numpy(dtype=float)
budget_l2 = float(np.quantile(finite_l2, 0.95)) if finite_l2.size else 1.0

vcs_rows = []
for method in method_order:
    sub = df[df["method_label"] == method]
    distances = sub["l2_distance"].to_numpy(dtype=float)
    distances[~sub["success"].to_numpy(dtype=bool)] = np.inf
    distances[~np.isfinite(distances)] = np.inf
    vcs_rows.append({"method": method, "vcs_l2": validity_closeness_score(distances, budget=budget_l2)})

vcs_summary = pd.DataFrame(vcs_rows).set_index("method").reindex(method_order)
display(vcs_summary.style.format("{:.3f}").background_gradient(cmap="Purples", axis=0))

plt.figure(figsize=(8, 4))
sns.barplot(x=vcs_summary.index, y=vcs_summary["vcs_l2"].values)
plt.title(f"Joint validity-closeness score (L2 budget = {budget_l2:.3f})")
plt.xlabel("")
plt.ylabel("score")
plt.xticks(rotation=25)
plt.show()


## 7. Qualitative examples

We visualize a few tasks that have successful counterfactuals for several methods. Each figure shows the original digit, one counterfactual per method, and the corresponding absolute-difference maps.

In [ ]:
successful_counts = (
    df.groupby(["query_idx", "target_class"])["success"]
    .sum()
    .sort_values(ascending=False)
)
selected_tasks = successful_counts[successful_counts >= max(2, len(method_order) // 2)].head(3).index.tolist()

for query_idx, target_class in selected_tasks:
    task_df = df[(df["query_idx"] == query_idx) & (df["target_class"] == target_class)].copy()
    task_df = task_df.set_index("method_label")
    ref = task_df.iloc[0]
    x_orig = ref[x_orig_cols].to_numpy(dtype=float).reshape(28, 28)
    source_class = int(ref["source_class"])
    y_true = int(ref["y_true"]) if "y_true" in ref.index and not pd.isna(ref["y_true"]) else None

    fig, axes = plt.subplots(2, len(method_order) + 1, figsize=(3 * (len(method_order) + 1), 5.5))
    axes[0, 0].imshow(x_orig, cmap="gray", vmin=0, vmax=1)
    axes[0, 0].set_title(f"Original\npred={source_class}" + (f"\ntrue={y_true}" if y_true is not None else ""))
    axes[1, 0].axis("off")

    for col, method in enumerate(method_order, start=1):
        ax_img = axes[0, col]
        ax_diff = axes[1, col]
        if method in task_df.index:
            row = task_df.loc[method]
            x_cf = row[x_cf_cols].to_numpy(dtype=float)
            if np.isfinite(x_cf).all():
                x_cf_img = x_cf.reshape(28, 28)
                ax_img.imshow(x_cf_img, cmap="gray", vmin=0, vmax=1)
                ax_diff.imshow(np.abs(x_cf_img - x_orig), cmap="magma")
                ax_img.set_title(f"{method}\nsuccess={bool(row['success'])}\npred={int(row['y_cf']) if not pd.isna(row['y_cf']) else 'NA'}")
            else:
                ax_img.text(0.5, 0.5, "no CF", ha="center", va="center")
                ax_diff.axis("off")
                ax_img.set_title(method)
        else:
            ax_img.text(0.5, 0.5, "missing", ha="center", va="center")
            ax_img.set_title(method)
            ax_diff.axis("off")
        ax_img.axis("off")
        if ax_diff.has_data():
            ax_diff.set_title("|CF - x|")
            ax_diff.axis("off")

    fig.suptitle(f"query_idx={query_idx}, source={source_class}, target={target_class}")
    plt.tight_layout()
    plt.show()
